# Audio Deepfake Detector — clean restart

This notebook trains a **bonafide vs spoof** classifier on **ASVspoof 2019 Logical Access (LA)**.

Run every cell in order with `Shift+Enter`. The whole workflow is:

1. Imports and settings
2. Find the dataset folders
3. Read the protocol labels
4. Extract audio features
5. Train one baseline model (Random Forest)
6. Save the trained model for the Flask app
7. Test your own audio files
8. Optional: evaluate the official eval split

> **One rule for the whole project:** training and prediction must both call
> `extract_features()` from `audio_features.py`. Never copy feature code into the notebook.

## 1. Imports and settings

The ASVspoof dataset is 20+ GB and lives **outside** this project folder
(`F:\minor project\LA`). `config.py` holds that path in one place — change it there
if the dataset ever moves.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# Make audio_features.py / config.py importable no matter where the kernel starts.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from audio_features import (
    CHUNK_SECONDS,
    N_MFCC,
    SAMPLE_RATE,
    extract_features_from_file,
    feature_names,
)

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
DATASET_ROOT = config.DATASET_ROOT
MODEL_DIR = config.MODELS_DIR
RESULTS_DIR = config.RESULTS_DIR

MODEL_PATH = config.MODEL_PATH
MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
THRESHOLD = 0.65        # spoof probability >= threshold -> "AI-GENERATED / SPOOF"
MAX_TRAIN_FILES = None  # e.g. 4000 for a quick end-to-end test; None uses all files

print("Dataset root :", DATASET_ROOT)
print("Model will be saved to:", MODEL_PATH)
print("Feature count:", len(feature_names()))

## 2. Find dataset folders

Expected folders: `ASVspoof2019_LA_train/flac`, `ASVspoof2019_LA_dev/flac`,
`ASVspoof2019_LA_eval/flac` and `ASVspoof2019_LA_cm_protocols`.
The search is recursive, so a nested layout still works.

In [ ]:
def find_named_folder(name):
    """Find a dataset folder by name anywhere under DATASET_ROOT."""
    matches = [path for path in DATASET_ROOT.rglob(name) if path.is_dir()]
    if not matches:
        raise FileNotFoundError(
            f"Could not find {name} under {DATASET_ROOT}. "
            "Check DATASET_ROOT in config.py."
        )
    return matches[0]


def get_split_audio_dir(split):
    folder = find_named_folder(f"ASVspoof2019_LA_{split}")
    audio_dir = folder / "flac"
    if not audio_dir.exists():
        raise FileNotFoundError(f"Missing audio folder: {audio_dir}")
    return audio_dir


PROTOCOL_DIR = find_named_folder("ASVspoof2019_LA_cm_protocols")
print("Protocol folder:", PROTOCOL_DIR)

for split in ("train", "dev", "eval"):
    audio_dir = get_split_audio_dir(split)
    file_count = sum(1 for _ in audio_dir.glob("*.flac"))
    print(f"{split:5s} audio folder: {audio_dir}  ({file_count:,} files)")

## 3. Read protocol labels

The last column of each protocol line is the label: **bonafide → 0**, **spoof → 1**.
The second column is the audio file ID, so the file is `<audio_dir>/<id>.flac`.

In [ ]:
def find_cm_protocol(split):
    """Locate the countermeasure protocol file for one split."""
    matches = [
        path for path in PROTOCOL_DIR.glob("*.txt")
        if ".cm." in path.name.lower() and split in path.name.lower()
    ]
    if not matches:
        raise FileNotFoundError(f"Could not find the CM {split} protocol in {PROTOCOL_DIR}")
    return matches[0]


def read_protocol(split):
    """Return a DataFrame with audio_path, label (0/1) and label_name."""
    protocol_path = find_cm_protocol(split)
    audio_dir = get_split_audio_dir(split)

    records = []
    missing = 0

    with protocol_path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            parts = line.strip().split()
            if len(parts) < 2:
                continue

            label_name = parts[-1].lower()
            if label_name not in {"bonafide", "spoof"}:
                continue

            audio_path = audio_dir / f"{parts[1]}.flac"
            if not audio_path.exists():
                missing += 1
                continue

            records.append({
                "audio_path": str(audio_path),
                "label": 0 if label_name == "bonafide" else 1,
                "label_name": label_name,
            })

    frame = pd.DataFrame(records)
    if frame.empty:
        raise RuntimeError(f"No labelled audio files found using {protocol_path}")

    bonafide = int((frame["label"] == 0).sum())
    spoof = int((frame["label"] == 1).sum())
    print(f"{split:5s}: {len(frame):,} files  ({bonafide:,} bonafide / {spoof:,} spoof)")
    if missing:
        print(f"       warning: {missing:,} protocol entries had no matching .flac file")

    return frame


train_df = read_protocol("train")
dev_df = read_protocol("dev")

## 4. Extract audio features

Each file becomes one **90-value vector** from `audio_features.py`:
MFCC and delta-MFCC (mean and std per coefficient) plus spectral centroid, bandwidth,
rolloff, zero-crossing rate and RMS (mean and std).

Files that cannot be decoded are skipped and reported instead of stopping the run.

In [ ]:
def subsample(frame, max_rows, random_state=RANDOM_STATE):
    """Shrink a split while keeping the original bonafide/spoof ratio."""
    if max_rows is None or max_rows >= len(frame):
        return frame.reset_index(drop=True)

    fraction = max_rows / len(frame)
    keep = []
    for _, group in frame.groupby("label"):
        count = max(1, round(len(group) * fraction))
        keep.extend(group.sample(n=count, random_state=random_state).index)

    return (
        frame.loc[keep]
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )


def make_feature_matrix(frame, split_name):
    """Extract one feature vector per audio file."""
    rows = []
    labels = []
    failed = []

    for row in tqdm(frame.itertuples(index=False), total=len(frame), desc=f"{split_name} features"):
        try:
            rows.append(extract_features_from_file(row.audio_path))
            labels.append(row.label)
        except Exception as exc:
            failed.append((row.audio_path, str(exc)))

    if not rows:
        raise RuntimeError(f"No features could be extracted for {split_name}.")

    X = np.vstack(rows)
    y = np.asarray(labels, dtype=np.int32)

    print(f"{split_name:5s}: feature matrix {X.shape}, skipped {len(failed)} file(s)")
    if failed:
        print("       first skipped file:", failed[0])

    return X, y


train_work = subsample(train_df, MAX_TRAIN_FILES)
print(f"Training on {len(train_work):,} files")

X_train, y_train = make_feature_matrix(train_work, "train")
X_dev, y_dev = make_feature_matrix(dev_df, "dev")

## 5. Train one baseline model

A single Random Forest inside a pipeline. The pipeline imputes any missing feature
values first, so one corrupt vector cannot break training.

`class_weight="balanced_subsample"` is used because the dataset is roughly **9:1 spoof
to bonafide** — without it the model would simply predict "spoof" almost always.

In [ ]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=2,
    )),
])

model.fit(X_train, y_train)


def evaluate(y_true, probabilities, threshold, name):
    """Print accuracy, precision, recall, F1 and the confusion matrix."""
    predictions = (probabilities >= threshold).astype(int)

    print(f"{name} at threshold {threshold}")
    print(f"  accuracy : {accuracy_score(y_true, predictions):.4f}")
    print(f"  precision: {precision_score(y_true, predictions, zero_division=0):.4f}")
    print(f"  recall   : {recall_score(y_true, predictions, zero_division=0):.4f}")
    print(f"  F1       : {f1_score(y_true, predictions, zero_division=0):.4f}")

    print("\nClassification report:")
    print(classification_report(
        y_true, predictions,
        target_names=["BONAFIDE", "SPOOF"],
        zero_division=0,
    ))

    matrix = confusion_matrix(y_true, predictions)
    print("Confusion matrix (rows = actual, columns = predicted):")
    print(matrix)

    return predictions, matrix


dev_probabilities = model.predict_proba(X_dev)[:, 1]
dev_predictions, dev_matrix = evaluate(y_dev, dev_probabilities, THRESHOLD, "Dev")

### Confusion matrix figure

Saved to `results/confusion_matrix.png` so you can drop it straight into your report.

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4))
ax.imshow(dev_matrix, cmap="Blues")

ax.set_xticks([0, 1], ["BONAFIDE", "SPOOF"])
ax.set_yticks([0, 1], ["BONAFIDE", "SPOOF"])
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
ax.set_title("Dev confusion matrix")

for row in range(2):
    for column in range(2):
        value = dev_matrix[row, column]
        ax.text(
            column, row, f"{value:,}",
            ha="center", va="center",
            color="white" if value > dev_matrix.max() / 2 else "#101828",
        )

fig.tight_layout()
figure_path = RESULTS_DIR / "confusion_matrix.png"
fig.savefig(figure_path, dpi=150)
plt.show()
print("Saved:", figure_path)

## 6. Save the trained model

The Flask app loads this one file. Do not change the feature extraction code between
training and prediction.

In [ ]:
model_bundle = {
    "model": model,
    "threshold": THRESHOLD,
    "sample_rate": SAMPLE_RATE,
    "chunk_seconds": CHUNK_SECONDS,
    "n_mfcc": N_MFCC,
    "n_features": len(feature_names()),
    "feature_names": feature_names(),
    "feature_description": (
        "mean/std of MFCC and delta-MFCC plus "
        "spectral centroid, bandwidth, rolloff, zero-crossing rate and RMS"
    ),
    "trained_on_files": int(len(y_train)),
    "dev_f1": float(f1_score(y_dev, dev_predictions, zero_division=0)),
}

joblib.dump(model_bundle, MODEL_PATH)
print("Saved model:", MODEL_PATH.resolve())
print(f"Trained on {model_bundle['trained_on_files']:,} files, "
      f"dev F1 = {model_bundle['dev_f1']:.4f}")

## 7. Test your own audio files

Put test files in `data/` (or anywhere) and list them below. These files must **not** be
part of the training dataset — record your own voice, or download a TTS sample.

This is the honest "completely new audio" test required for the project.

In [ ]:
from audio_features import load_model, predict_audio_file

bundle = load_model(MODEL_PATH)

test_files = [
    # Path("data/real/my_voice.wav"),
    # Path("data/ai/generated_voice.mp3"),
]

for audio_path in test_files:
    if not Path(audio_path).exists():
        print(f"Not found: {audio_path}")
        continue
    try:
        result = predict_audio_file(audio_path, bundle)
        print(
            f"{Path(audio_path).name:35s} -> {result['display_label']:22s} "
            f"spoof score {result['spoof_score']:6.2f}%  "
            f"({result['chunks_analysed']} chunk(s), {result['duration_seconds']}s)"
        )
    except Exception as exc:
        print(f"{Path(audio_path).name:35s} -> ERROR: {exc}")

## 8. Optional: evaluate the official eval split

Run this only after the train/dev pipeline works. It extracts features for 71,237 files,
so it takes considerably longer.

In [ ]:
RUN_OFFICIAL_EVAL = False

if RUN_OFFICIAL_EVAL:
    eval_df = read_protocol("eval")
    X_eval, y_eval = make_feature_matrix(eval_df, "eval")
    eval_probabilities = model.predict_proba(X_eval)[:, 1]
    evaluate(y_eval, eval_probabilities, THRESHOLD, "Eval")
else:
    print("Skipped. Set RUN_OFFICIAL_EVAL = True to run the official evaluation.")